# Hangman letter analysis

Letter-focused tables split out from `strategy_explorer.ipynb`: where a letter
gets guessed, what a set of guessed letters resolves to, letter frequency by
position, and common n-grams. Same `strategy.py` / `strategy_view.py` backend and
the `LENGTH` / `WEIGHTING` / `STRATEGY` knobs below (each table also takes explicit
args, so you can override per call).

In [1]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'strategy.py').exists():
            return c
    raise FileNotFoundError('Could not find strategy.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import strategy
import strategy_view as sview
import wordlist

pd.set_option('display.max_rows', 200)   # show up to 200 rows before pandas truncates to a head/tail preview

print(f'imported strategy + strategy_view from {math_dir}')

imported strategy + strategy_view from /Users/patrickliscio/My Documents/Pycharm Projects/Ballpark-Figures/hangman/math


## Defaults

These knobs are the defaults the tables below use. `WEIGHTING` sets the frequency
weighting for **every** table. `STRATEGY` / `BUDGET` / `FILL_CAPPED` affect only the
strategy-dependent tables (`guesses_of_letter*`, `letters_summary*`); the position
and n-gram tables are **wordlist-only**. `LENGTH` is the default length for the
single-length tables. Tables that sweep every length build their own per-length
views, so unsolved `optimal`/`winprob` lengths are skipped cleanly.

In [55]:
LENGTH    = 5           # <-- default word length for single-length tables
WEIGHTING = "uniform"   # <-- uniform, zipf, greedy, best_opener, greedy_word, best_opener_word
STRATEGY  = "winprob"   # <-- greedy, greedy_best, optimal, winprob (strategy-dependent tables only)
BUDGET    = 5        # <-- winprob only: miss budget k (None = most permissive solved)
FILL_CAPPED = True      # <-- optimal only: finish cap-truncated branches with greedy

print(f'defaults -> length {LENGTH}, weighting {WEIGHTING!r}, strategy {STRATEGY!r}')

DICTIONARY = "us"   # <-- dictionary variant for EVERY table below: "us" (American) or "gb" (British)

# Route all tables through DICTIONARY without editing each function: default
# strategy_view.make_strategy's `variant` to DICTIONARY (read at call time). Idempotent.
if not getattr(sview, "_variant_wrapped", False):
    sview._make_strategy_orig = sview.make_strategy
    def _make_strategy(*a, **k):
        k.setdefault("variant", DICTIONARY)
        return sview._make_strategy_orig(*a, **k)
    sview.make_strategy = _make_strategy
    sview._variant_wrapped = True
print(f'dictionary variant: {DICTIONARY!r}  (us = American, gb = British)')

defaults -> length 5, weighting 'uniform', strategy 'winprob'
dictionary variant: 'us'  (us = American, gb = British)


## Where letters are guessed (single length)

`guesses_of_letter(letters, length)` finds **every** situation -- a reachable board
with **nonzero probability** under the weighting -- at `length` where the strategy's
next guess is one of `letters` (a string like `'u'` or `'aeiou'`), sorted by **level**
(depth = guesses already made) then by **freq_words** (descending). Columns:

- **level** -- guesses already made when this board is reached (0 = opener),
- **letter** -- which of `letters` the strategy guesses there,
- **board** / **missed** -- the revealed board and the missed letters there,
- **n_words** -- words consistent with that board,
- **freq_words** -- frequency-weighted words (`fmass / total_freq_L * N_L`); for a
  single length this is just a scaling of the raw frequency mass,
- **sample** -- a few example candidate words at that board,
- **hit_sample** -- a few candidates that actually **contain that letter** -- the
  words the guess would reveal (a hit).

Boards whose words all weigh 0 under the weighting are skipped (nonzero-prob only).
With several letters, rows interleave by level/freq -- sort by `letter` to group them.
Uses the current `STRATEGY`/`WEIGHTING`/`BUDGET`/`FILL_CAPPED`.

In [60]:
LETTERS  = 'a'       # <-- one or more letters to locate (e.g. 'u' or 'aeiou')
WORD_LEN = LENGTH    # <-- word length (defaults to the LENGTH picked at the top)

def guesses_of_letter(letters, length, *, strategy=STRATEGY, weighting=WEIGHTING,
                      budget=BUDGET, fill_capped=FILL_CAPPED, max_level=40,
                      verbose=True):
    """Every situation at `length` where the strategy's next guess is one of `letters`
    (a string like 'u' or 'aeiou') and the board has nonzero probability under the
    weighting, sorted by level (depth) then freq_words (descending). Columns: level,
    letter (which letter is guessed there), board, missed, n_words, freq_words
    (= fmass / total_freq_L * N_L), sample (any candidates there), hit_sample
    (candidates that CONTAIN that letter -- the words the guess would reveal).
    Determined branches (no guess) and zero-frequency boards are skipped.
    `verbose=False` silences the skip/empty notes (used by the all-lengths sweep)."""
    letterset = {c for c in letters.lower() if c.isalpha()}
    cols = ['level', 'letter', 'board', 'missed', 'n_words', 'freq_words', 'sample', 'hit_sample']
    s = sview.make_strategy(strategy, length, weighting=weighting, budget=budget,
                            fill_capped=fill_capped)
    if not s.available:
        if verbose:
            print('not available:', s.note)
        return pd.DataFrame(columns=cols)
    total = float(s.word_freq().sum())
    scale = (s.g.N / total) if total else 0.0          # freq share -> effective words
    rows = []
    for level in range(max_level + 1):
        nodes = s.states_at_depth(level)
        if not nodes:                                   # tree bottomed out
            break
        for st in nodes:
            if st['determined'] or st['guess'] not in letterset or st['fmass'] == 0.0:
                continue
            g_letter = st['guess']
            missed = ''.join(st['missed'])
            cand = s.state(st['board'], missed).get('candidates', [])
            hit = [w for w in cand if g_letter in w]     # words this guess reveals
            rows.append({'level': level, 'letter': g_letter, 'board': st['board'],
                         'missed': missed, 'n_words': st['n'],
                         'freq_words': st['fmass'] * scale,
                         'sample': ' '.join(st['sample'][:5]),
                         'hit_sample': ' '.join(hit[:5])})
    if not rows:
        if verbose:
            shown = ''.join(sorted(letterset))
            print(f'{shown!r} is never the next guess at a nonzero-probability board '
                  f'(length {length}, {s.label})')
        return pd.DataFrame(columns=cols)
    return (pd.DataFrame(rows)
              .sort_values(['level', 'freq_words'], ascending=[True, False])
              .reset_index(drop=True))

res = guesses_of_letter(LETTERS, WORD_LEN).sort_values(by="n_words", ascending=False)
shown = ''.join(sorted({c for c in LETTERS.lower() if c.isalpha()}))
print(f'{shown!r} at length {WORD_LEN}: {len(res)} situations  [{STRATEGY} / {WEIGHTING}]')
with pd.option_context('display.max_colwidth', None):
    display(res.head(200))

'a' at length 5: 88 situations  [winprob / uniform]


,level,letter,board,missed,n_words,freq_words,sample,hit_sample
0,1,a,_____,e,1856,1856.0,about which would first think,about again start today small
1,1,a,____e,,356,356.0,those while state since house,state place maybe large cause
8,3,a,___es,r,137,137.0,times makes games comes takes,makes games takes names taxes
9,3,a,_e___,rs,94,94.0,being death media legal began,death media legal began heavy
25,4,a,___ed,rs,83,83.0,moved loved named lived liked,named faced naked acted aimed
10,3,a,___er,i,79,79.0,other after under power water,after water later paper anger
2,2,a,__e__,s,65,65.0,their great check clear field,great clear break ahead clean
48,5,a,___e_,dors,38,38.0,given taken chief quiet panel,taken panel angel label alien
3,2,a,re___,,27,27.0,ready reach reply relax reads,ready reach relax reads react
4,2,a,_e_r_,,24,24.0,years heart heard learn weird,years heart heard learn tears


## Where letters are guessed (all lengths)

`guesses_of_letter_all_lengths(letters)` is the same as above but swept over **every
word length**: every situation (at any length) where the strategy's next guess is one
of `letters`, sorted by **level**, then **length**, then **freq_words** (descending).
It adds a `length` column; because `freq_words` is the cross-length-comparable metric
(`fmass / total_freq_L * N_L`), situations from different lengths sit on the same scale
and sort together. The `letter` column shows which letter is guessed. Lengths with no
committed solve are skipped. Uses the current `STRATEGY`/`WEIGHTING`/`BUDGET`/`FILL_CAPPED`.

In [61]:
def guesses_of_letter_all_lengths(letters, lengths=range(1, 21), *,
                                  strategy=STRATEGY, weighting=WEIGHTING,
                                  budget=BUDGET, fill_capped=FILL_CAPPED):
    """`guesses_of_letter` swept over every word length: every situation (any length)
    where the strategy's next guess is one of `letters` and the board has nonzero
    probability, sorted by level, then length, then freq_words (descending). Adds a
    `length` column; `freq_words` is the cross-length-comparable metric
    (fmass / total_freq_L * N_L), so situations from different lengths sort together.
    The `letter` column shows which letter is guessed."""
    cols = ['level', 'length', 'letter', 'board', 'missed', 'n_words', 'freq_words',
            'sample', 'hit_sample']
    parts = []
    for L in lengths:
        df = guesses_of_letter(letters, L, strategy=strategy, weighting=weighting,
                               budget=budget, fill_capped=fill_capped, verbose=False)
        if len(df):
            df.insert(1, 'length', L)
            parts.append(df)
    if not parts:
        shown = ''.join(sorted({c for c in letters.lower() if c.isalpha()}))
        print(f'{shown!r} is never guessed at any length  [{strategy} / {weighting}]')
        return pd.DataFrame(columns=cols)
    return (pd.concat(parts, ignore_index=True)
              .sort_values(['level', 'length', 'freq_words'],
                           ascending=[True, True, False])
              .reset_index(drop=True))

res_all = guesses_of_letter_all_lengths(LETTERS).sort_values(by="n_words", ascending=False)
shown = ''.join(sorted({c for c in LETTERS.lower() if c.isalpha()}))
print(f'{shown!r}, all lengths: {len(res_all)} situations  [{STRATEGY} / {WEIGHTING}]')
with pd.option_context('display.max_colwidth', None):
    display(res_all.head(200))

'a', all lengths: 2311 situations  [winprob / uniform]


,level,length,letter,board,missed,n_words,freq_words,sample,hit_sample
2,0,4,a,____,,2120,2120.0,that with this have from,that have what also than
3,1,5,a,_____,e,1856,1856.0,about which would first think,about again start today small
15,2,6,a,______,ei,897,897.0,should around always school though,around always thanks almost across
1,0,3,a,___,,640,640.0,the and for you was,and was are all can
36,2,7,a,_______,ei,618,618.0,through company thought support country,company program account usually forward
58,2,8,a,________,ei,454,454.0,actually probably although football products,actually probably although football standard
169,3,7,a,____ing,,385,385.0,looking working playing talking meeting,playing talking reading amazing leading
4,1,5,a,____e,,356,356.0,those while state since house,state place maybe large cause
59,2,8,a,______e_,i,333,333.0,daughter followed computer compared supposed,daughter compared transfer advanced attorney
108,3,6,a,____er,i,187,187.0,number matter father mother summer,matter father player answer cancer


In [14]:
consonants = 'bcdfghjklmnpqrstvwxz'
res2 = guesses_of_letter_all_lengths(consonants)
with pd.option_context('display.max_colwidth', None):
    display(res2.head(200))

,level,length,letter,board,missed,n_words,freq_words,sample,hit_sample
0,0,17,t,_________________,,18,18.0,interdisciplinary indistinguishable telecommunication institutionalized misunderstandings,interdisciplinary indistinguishable telecommunication institutionalized misunderstandings
1,0,18,t,__________________,,3,3.0,telecommunications disproportionately characteristically,telecommunications disproportionately characteristically
2,0,19,c,___________________,,1,1.0,counterintelligence,counterintelligence
3,1,2,t,a_,,7,7.0,as at an am ad,at
4,1,3,t,_a_,,136,136.0,was can has had way,tax eat fat cat sat
5,1,3,p,a__,,36,36.0,and are all any ago,app amp apt ape alp
6,1,3,b,_aa,,1,1.0,baa,baa
7,1,4,s,_a__,,400,400.0,have make back want said,said last same days case
8,1,4,r,__a_,,199,199.0,that what than year team,year real read near road
9,1,4,t,___a,,42,42.0,idea beta yoga visa soda,beta meta tuna vita zeta


## Summary for a set of guessed letters

`letters_summary(letters)` takes a collection of `letters` (a string; case, dupes
and non-letters are ignored) treated as **already guessed**, and reports over the
current `LENGTH`'s word list a word **count** and **total frequency** (the
weighting's `fmass`, and its % of all words/frequency) for these rows:

- **only these letters** — words spelled entirely from `letters` (so fully revealed
  once all are guessed);
- **uniquely determined after** — words whose revealed pattern of `letters` (which of
  them sit at which positions, blanks elsewhere) is shared by no other word, so
  guessing `letters` already pins the word down;
- **misses after = k** — one row per number of **further misses** the strategy makes
  finishing the word once `letters` have all been guessed (`k = 0` is the miss-free
  case). A trailing **unsolved** row appears if any word can't be solved at all.

The `misses after = k` rows **partition** every solvable word; the first two rows are
(overlapping) subsets of `misses after = 0` (only-these ⊆ uniquely-determined ⊆
miss-free).

> **Miss-count semantics (force-guessed).** We **force-guess** all of `letters` — reaching
> the board where each is revealed (or known absent; absent ones are misses *during*
> that phase, which don't count) — then **finish greedily** and count the further misses
> (a miss still narrows: the letter is now known absent). For the `greedy`/`greedy_best`
> strategies this greedy finish IS the strategy; for `optimal`/`winprob` the continuation
> from an arbitrary forced state isn't in the committed tree, so the finish is greedy
> (greedy-below-`L`, the same convention as `FILL_CAPPED`). Say if you'd rather it use the
> selected strategy's own play order instead of a forced-`L`-first model.

In [6]:
GUESSED_LETTERS = 'aeiounslrt'   # <-- letters treated as already guessed (case/dupes/non-letters ignored)

def _greedy_misses_after(g, freq, wi, Lidx, guessed0):
    """Number of misses a GREEDY play makes finishing word `wi` after force-guessing
    the letters `Lidx` (bitmask `guessed0`) -- how many of greedy's remaining guesses
    are absent from `wi`. 0 == solved miss-free from the L-state."""
    m = g.masks
    S = np.arange(g.N)
    for c in Lidx:                                  # candidates sharing wi's L-reveal
        S = S[m[S, c] == m[wi, c]]
    guessed, misses = guessed0, 0
    while S.shape[0] > 1:
        lf = freq[S] @ g.present[S]                 # greedy score: freq-weighted presence
        gg = guessed
        while gg:                                   # mask already-guessed letters
            c0 = (gg & -gg).bit_length() - 1
            lf[c0] = -1.0
            gg &= gg - 1
        c = int(lf.argmax())
        if lf[c] < 0:
            break
        if not m[wi, c]:                            # greedy guesses a letter absent from wi -> miss
            misses += 1
        S = S[m[S, c] == m[wi, c]]                  # a miss still narrows (letter known absent)
        guessed |= (1 << c)
    return misses

def _letter_stats(s, letters):
    """Per-word arrays for guessed `letters` (aligned to `s.g.words`): only_L (spelled
    only from them), uniq (uniquely determined once they're guessed), misses_after
    (# further misses the greedy finish makes after force-guessing them; -1 for words
    the strategy can't solve at all)."""
    g, freq, words = s.g, s.word_freq(), s.g.words
    Lset = {c for c in letters.lower() if c.isalpha()}
    Lidx = {ord(c) - 97 for c in Lset}
    guessed0 = 0
    for c in Lidx:
        guessed0 |= (1 << c)
    outside = [c for c in range(26) if c not in Lidx]
    only_L = (g.present[:, outside].sum(axis=1) == 0) if Lidx else np.zeros(g.N, bool)
    sigs = [''.join(ch if ch in Lset else '_' for ch in w) for w in words]
    counts = pd.Series(sigs).value_counts()
    uniq = np.array([counts[si] == 1 for si in sigs])
    status = s.per_word_status()                    # 1 capped / 2 lost -> unsolved
    misses_after = np.array([-1 if status[i] in (1, 2)
                             else _greedy_misses_after(g, freq, i, Lidx, guessed0)
                             for i in range(g.N)], dtype=int)
    return only_L, uniq, misses_after

def _letter_rows(only_L, uniq, misses_after):
    """Ordered (label, boolean-mask) rows: the two descriptive categories, then one
    row per # of further misses (0, 1, ... up to the max seen), then unsolved if any.
    The `misses after = k` rows PARTITION the solvable words; the first two are
    (overlapping) subsets of `misses after = 0`."""
    solved = misses_after >= 0
    rows = [('only these letters', only_L), ('uniquely determined after', uniq)]
    kmax = int(misses_after[solved].max()) if solved.any() else -1
    for k in range(kmax + 1):
        rows.append((f'misses after = {k}', misses_after == k))
    if (~solved).any():
        rows.append(('unsolved (capped/lost)', ~solved))
    return rows

def letters_summary(letters, *, strategy=STRATEGY, length=LENGTH, weighting=WEIGHTING,
                    budget=BUDGET, fill_capped=FILL_CAPPED):
    """Per-category count + total frequency (the weighting's fmass) of length-`length`
    words for guessed `letters`. Rows: spelled-only-from, uniquely-determined, then one
    row per # of further misses the strategy makes after force-guessing them
    (`misses after = k`; k=0 is the miss-free case). See markdown for semantics."""
    s = sview.make_strategy(strategy, length, weighting=weighting, budget=budget,
                            fill_capped=fill_capped)
    if not s.available:
        print('not available:', s.note)
        return pd.DataFrame()
    freq = s.word_freq()
    rows = _letter_rows(*_letter_stats(s, letters))
    tot_n, tot_f = s.g.N, float(freq.sum())
    df = pd.DataFrame([{'category': name, 'n_words': int(m.sum()),
                        'total_freq': round(float(freq[m].sum()), 1),
                        'pct_words': round(100 * int(m.sum()) / tot_n, 2),
                        'pct_freq': round(100 * float(freq[m].sum()) / tot_f, 2) if tot_f else 0.0}
                       for name, m in rows])
    shown = ''.join(sorted({c for c in letters.lower() if c.isalpha()}))
    print(f"letters = {shown!r}  |  length {length}  |  {s.label}")
    print(f"totals: {tot_n:,} words, freq {tot_f:,.1f}")
    return df

display(letters_summary(GUESSED_LETTERS))

letters = 'aeilnorstu'  |  length 5  |  optimal (uniform, cap 7*+greedy)
totals: 3,495 words, freq 3,495.0


,category,n_words,total_freq,pct_words,pct_freq
0,only these letters,301,301.0,8.61,8.61
1,uniquely determined after,1595,1595.0,45.64,45.64
2,misses after = 0,2310,2310.0,66.09,66.09
3,misses after = 1,691,691.0,19.77,19.77
4,misses after = 2,286,286.0,8.18,8.18
5,misses after = 3,132,132.0,3.78,3.78
6,misses after = 4,46,46.0,1.32,1.32
7,misses after = 5,19,19.0,0.54,0.54
8,misses after = 6,9,9.0,0.26,0.26
9,misses after = 7,2,2.0,0.06,0.06


## Summary for a set of guessed letters (all lengths)

`letters_summary_all_lengths(letters)` is the same summary (same rows, including one
per `misses after = k`) swept over **every word length**. Word counts (`n_words`) sum
across lengths; the frequency column uses the **cross-length metric** `freq_words`
(each word contributes `fmass / total_freq_L * N_L`, as in the guess tables), so the
lengths sit on one "effective words" scale and add up sensibly. Under `uniform`,
`freq_words == n_words`. Both `pct_words` and `pct_freq` are shares of the total over
all lengths (the total `freq_words` equals the total word count by construction). The
`misses after = k` rows span `k = 0` up to the max across all lengths. Lengths with no
committed solve are skipped.

In [7]:
def letters_summary_all_lengths(letters, lengths=range(1, 21), *,
                                strategy=STRATEGY, weighting=WEIGHTING,
                                budget=BUDGET, fill_capped=FILL_CAPPED):
    """`letters_summary` aggregated over every word length. Same rows (per # of
    further misses); `n_words` sums the raw counts and `freq_words` sums the
    cross-length metric fmass/total_freq_L * N_L (== n_words under uniform). Lengths
    with no committed solve are skipped."""
    per, tot_n, used, kmax, any_unsolved = [], 0, [], -1, False
    for L in lengths:
        s = sview.make_strategy(strategy, L, weighting=weighting, budget=budget,
                                fill_capped=fill_capped)
        if not s.available:
            continue
        used.append(L)
        freq = s.word_freq()
        total = float(freq.sum())
        scale = (s.g.N / total) if total else 0.0    # freq share -> effective words
        tot_n += s.g.N
        only_L, uniq, misses_after = _letter_stats(s, letters)
        solved = misses_after >= 0
        if solved.any():
            kmax = max(kmax, int(misses_after[solved].max()))
        any_unsolved = any_unsolved or bool((~solved).any())
        per.append((freq, scale, only_L, uniq, misses_after))
    if not used:
        print(f'no committed solve at any length  [{strategy} / {weighting}]')
        return pd.DataFrame()

    labels = (['only these letters', 'uniquely determined after']
              + [f'misses after = {k}' for k in range(kmax + 1)]
              + (['unsolved (capped/lost)'] if any_unsolved else []))
    n_words = {lab: 0 for lab in labels}
    freq_words = {lab: 0.0 for lab in labels}
    def add(lab, mask, freq, scale):
        n_words[lab] += int(mask.sum())
        freq_words[lab] += float(freq[mask].sum()) * scale
    for freq, scale, only_L, uniq, misses_after in per:
        add('only these letters', only_L, freq, scale)
        add('uniquely determined after', uniq, freq, scale)
        for k in range(kmax + 1):
            add(f'misses after = {k}', misses_after == k, freq, scale)
        if any_unsolved:
            add('unsolved (capped/lost)', misses_after < 0, freq, scale)

    df = pd.DataFrame([{'category': lab, 'n_words': n_words[lab],
                        'freq_words': round(freq_words[lab], 1),
                        'pct_words': round(100 * n_words[lab] / tot_n, 2),
                        'pct_freq': round(100 * freq_words[lab] / tot_n, 2)}  # total freq_words == tot_n
                       for lab in labels])
    shown = ''.join(sorted({c for c in letters.lower() if c.isalpha()}))
    print(f"letters = {shown!r}  |  lengths {used[0]}-{used[-1]} ({len(used)} solved)  |  {strategy} / {weighting}")
    print(f"totals: {tot_n:,} words (= total freq_words under the cross-length normalization)")
    return df

display(letters_summary_all_lengths(GUESSED_LETTERS))

letters = 'aeilnorstu'  |  lengths 1-20 (20 solved)  |  optimal / uniform
totals: 34,483 words (= total freq_words under the cross-length normalization)


,category,n_words,freq_words,pct_words,pct_freq
0,only these letters,1657,1657.0,4.81,4.81
1,uniquely determined after,26243,26243.0,76.10,76.10
2,misses after = 0,29589,29589.0,85.81,85.81
3,misses after = 1,2802,2802.0,8.13,8.13
4,misses after = 2,1082,1082.0,3.14,3.14
5,misses after = 3,521,521.0,1.51,1.51
6,misses after = 4,252,252.0,0.73,0.73
7,misses after = 5,126,126.0,0.37,0.37
8,misses after = 6,67,67.0,0.19,0.19
9,misses after = 7,29,29.0,0.08,0.08


## Letter frequency by position (all lengths)

`letter_position_table(letters)` — for a letter or **list of letters** (`'e'`, or
`'aeiou'`, or `['a','e']`), one row per **(length, position)** giving the proportion
of words, and of frequency (under `WEIGHTING`), whose character at that position is
**any** of `letters`. Positions are **1-indexed** (1 = first letter); proportions are
per-length (the denominator is that length's word / frequency total). Wordlist-only
(independent of `STRATEGY`); sorted by length then position. Under `uniform`,
`prop_freq == prop_words`.

In [8]:
POS_LETTERS = 'y'   # <-- one or more letters (e.g. 'aeiou'); counts positions holding ANY of them

def letter_position_table(letters, lengths=range(1, 21), *, weighting=WEIGHTING):
    """One row per (length, position): the proportion of length-`length` words -- and
    of their frequency (the weighting's fmass) -- whose character at that position is
    ANY of `letters` (a string like 'aeiou' or a list). Positions are 1-indexed
    (1 = first letter). Wordlist-only (independent of STRATEGY); sorted by length then
    position."""
    Lset = {c for c in ''.join(letters).lower() if c.isalpha()}
    Lidx = [ord(c) - 97 for c in Lset]
    rows = []
    for L in lengths:
        s = sview.make_strategy('greedy', L, weighting=weighting)
        if not s.available:
            continue
        g, freq = s.g, s.word_freq()
        totf = float(freq.sum())
        combo = np.zeros(g.N, dtype=g.masks.dtype)     # per-word: positions holding any listed letter
        for ci in Lidx:
            combo |= g.masks[:, ci]
        for p in range(L):
            has = ((combo >> np.uint32(p)) & np.uint32(1)).astype(bool)
            nw = int(has.sum())
            rows.append({'length': L, 'position': p + 1, 'n_words': nw,
                         'prop_words': round(nw / g.N, 4) if g.N else 0.0,
                         'prop_freq': round(float(freq[has].sum()) / totf, 4) if totf else 0.0})
    df = pd.DataFrame(rows).sort_values(['length', 'position']).reset_index(drop=True)
    print(f"letters {''.join(sorted(Lset))!r}  |  weighting {weighting}  |  {len(df)} (length, position) rows")
    return df

with pd.option_context('display.max_rows', None):
    display(letter_position_table(POS_LETTERS).head(55))

letters 'y'  |  weighting uniform  |  210 (length, position) rows


,length,position,n_words,prop_words,prop_freq
0,1,1,0,0.0000,0.0000
1,2,1,3,0.0612,0.0612
2,2,2,2,0.0408,0.0408
3,3,1,15,0.0234,0.0234
4,3,2,7,0.0109,0.0109
5,3,3,41,0.0641,0.0641
6,4,1,20,0.0094,0.0094
7,4,2,22,0.0104,0.0104
8,4,3,20,0.0094,0.0094
9,4,4,95,0.0448,0.0448


## Most common n-grams containing a letter (all lengths)

`letter_ngram_table(letter)` — the **bigrams, trigrams and (optionally) 4-grams** (adjacent runs) that
contain `letter`, ranked together by the **number of words** (pooled over all lengths) that
contain the n-gram; a word containing it more than once counts **once**. `pct_freq` is
the same but frequency-weighted (cross-length `fmass / total_freq_L * N_L`; `==
pct_words` under `uniform`). The `n` column is the n-gram length (2, 3, or 4). Set `INCLUDE_TRIGRAMS = False`
for bigrams only, or `INCLUDE_FOURGRAMS = True` to also mix in 4-grams. Sorted by word count descending,
**ties broken toward the shorter n-gram**; wordlist-only; uses the `GRAM_LETTER` knob. (Trigrams are rarer than bigrams, so they
naturally sit lower in the ranking. Length-1 words, with no n-gram, are still in the
denominator — it's "% of all words".)

In [9]:
GRAM_LETTER      = 'e'    # <-- letter the n-grams must contain
INCLUDE_TRIGRAMS = True   # <-- mix trigrams in with the bigrams
INCLUDE_FOURGRAMS = False  # <-- also mix in 4-grams

def letter_ngram_table(letter, lengths=range(1, 21), *, weighting=WEIGHTING,
                       include_trigrams=True, include_fourgrams=False):
    """N-grams (adjacent runs) containing `letter`, ranked by the % of words across
    all lengths that contain the n-gram -- a word with it more than once counts ONCE
    -- with the % of frequency-weighted words (cross-length fmass/total_freq_L * N_L;
    == % of words under uniform). Bigrams, plus trigrams (`include_trigrams`) and
    4-grams (`include_fourgrams`), ranked together. Sorted by word count descending,
    ties broken toward the SHORTER n-gram. `n` is the n-gram length. Wordlist-only
    (independent of STRATEGY)."""
    letter = letter.lower()
    sizes = [2] + ([3] if include_trigrams else []) + ([4] if include_fourgrams else [])
    nwords, fwords, tot_n = {}, {}, 0
    for L in lengths:
        s = sview.make_strategy('greedy', L, weighting=weighting)
        if not s.available:
            continue
        g, freq = s.g, s.word_freq()
        total = float(freq.sum())
        scale = (g.N / total) if total else 0.0       # cross-length effective words
        tot_n += g.N
        for i, w in enumerate(g.words):
            fwi = freq[i] * scale
            seen = {w[j:j + k] for k in sizes for j in range(len(w) - k + 1)
                    if letter in w[j:j + k]}
            for gram in seen:
                nwords[gram] = nwords.get(gram, 0) + 1
                fwords[gram] = fwords.get(gram, 0.0) + fwi
    if not nwords:
        print(f'no n-grams containing {letter!r}')
        return pd.DataFrame(columns=['ngram', 'n', 'n_words', 'pct_words', 'pct_freq'])
    df = pd.DataFrame([{'ngram': gram, 'n': len(gram), 'n_words': nwords[gram],
                        'pct_words': round(100 * nwords[gram] / tot_n, 3),
                        'pct_freq': round(100 * fwords[gram] / tot_n, 3)}
                       for gram in nwords]).sort_values(['n_words', 'n'], ascending=[False, True]).reset_index(drop=True)
    kinds = 'n=' + ','.join(str(k) for k in sizes)
    print(f"letter {letter!r}  |  {kinds}  |  weighting {weighting}  |  {tot_n:,} words, {len(df)} n-grams")
    return df

display(letter_ngram_table(GRAM_LETTER, include_trigrams=INCLUDE_TRIGRAMS, include_fourgrams=INCLUDE_FOURGRAMS).head(50))

letter 'e'  |  n=2,3  |  weighting uniform  |  34,483 words, 1189 n-grams


,ngram,n,n_words,pct_words,pct_freq
0,er,2,5084,14.743,14.743
1,ed,2,4044,11.728,11.728
2,es,2,4041,11.719,11.719
3,re,2,3798,11.014,11.014
4,te,2,3666,10.631,10.631
5,en,2,2957,8.575,8.575
6,le,2,2597,7.531,7.531
7,de,2,2128,6.171,6.171
8,ne,2,1751,5.078,5.078
9,se,2,1713,4.968,4.968


## Guess-order stats: set letters vs outside letters (all lengths)

Given a collection of `letters`, play the strategy under the **completion convention**
(keep guessing until the word is fully revealed: the strategy's real guesses in order,
then the word's remaining letters appended by **first appearance** in the word; letters
absent from the word that weren't guessed before it was determined don't appear), and
average two quantities over words, one row per length:

- **set_before_outside** *(request 1)* — the number of set-letter **occurrences**
  (multi-counted) revealed **before the first letter outside the set is guessed**;
- **pairs_outside_before_set** *(request 2)* — for each set letter α, the number of
  outside letters (β∉set) guessed **before** α; **and if α is never guessed for that
  word, every outside letter that *is* guessed counts** (so rare, seldom-guessed set
  letters are penalised, not rewarded). Summed over α∈set.

**The average is taken over the `WEIGHTING` distribution** — each word is weighted by its
frequency, cross-length-normalised to `N_L` per length (`freq × N_L/total_freq_L`, the
same "effective words" scheme as the other tables), so pairs on **common branches count
proportionally**. This is consistent with how the strategy plays (both use `WEIGHTING`),
and reduces to plain uniform-by-word when `WEIGHTING = 'uniform'`. The printed **OVERALL**
line gives the all-lengths average for both. Works for **any** `STRATEGY` (default
`optimal`).

> Fast: **one traversal of the decision tree per length** — the committed JSON tree for
> `optimal`/`winprob`, a live greedy recursion for the greedy views — collecting every
> word's guess path in a single DFS (well under a second per length).

In [10]:
def _each_word_order(s, cb):
    """Call cb(word, path) for every word; `path` is the strategy's root->leaf guess
    sequence (real guesses, in order). `path` is REUSED (append/pop) -- copy it if you
    keep a reference. Walks the committed JSON tree for optimal/winprob, or a live
    greedy recursion for the greedy views."""
    g = s.g
    if hasattr(s, 'tree'):                            # optimal / winprob
        def walk(node, path):
            if 'branches' not in node:                # leaf
                for w in ([node['word']] if 'word' in node else node.get('capped', node.get('lost', []))):
                    cb(w, path)
                return
            path.append(node['guess'])
            for child in node['branches'].values():
                walk(child, path)
            path.pop()
        walk(s.tree, [])
    else:                                             # greedy / greedy_best
        f = s.word_freq()
        oc = g._opener_index(getattr(s, '_opener', None))
        def walk(S, gb, path):
            if S.shape[0] <= 1:
                if S.shape[0] == 1:
                    cb(g.words[int(S[0])], path)
                return
            c = g._pick_o(S, gb, f, oc)               # greedy's guess (honours a forced opener)
            pats = g.masks[S, c]
            ngb = gb | (1 << c)
            path.append(chr(97 + int(c)))
            for pat in np.unique(pats):
                walk(S[pats == pat], ngb, path)
            path.pop()
        walk(np.arange(g.N), 0, [])

def _order_metrics(order, w, Lset):
    """(set_before_outside, pairs) for one word's guess `order` (distinct letters, in
    order) and the set `Lset`. Pairs, per set letter alpha: the number of outside
    letters guessed before alpha -- OR, if alpha is NEVER guessed for this word, every
    outside letter that IS guessed (so seldom-guessed set letters are penalised, not
    rewarded)."""
    first_out = next((k for k, ch in enumerate(order) if ch not in Lset), len(order))
    before = sum(w.count(ch) for ch in order[:first_out])
    beta_before = pairs = guessed_in = 0
    for ch in order:
        if ch in Lset:
            pairs += beta_before                      # outside letters guessed before this alpha
            guessed_in += 1
        else:
            beta_before += 1
    pairs += (len(Lset) - guessed_in) * beta_before   # never-guessed alphas: all outside count
    return before, pairs

def _accumulate(s, Lset):
    """ONE traversal (via _each_word_order) accumulating both metrics over all words.
    A word's guess `order` is its root->leaf guess path (the completion convention's
    real guesses) then the word's remaining letters by first appearance. Each word
    contributes its cross-length effective weight freq * N_L/total_freq_L (so it sums
    to N_L per length -- common branches count proportionally; == uniform-by-word when
    weighting='uniform'). Returns (sum_before, sum_pairs, sum_weight, n_words)."""
    g = s.g
    freq = s.word_freq()
    total = float(freq.sum())
    scale = (g.N / total) if total else 0.0
    idx = g._index
    tot = [0.0, 0.0, 0.0, 0]                           # sum_before, sum_pairs, sum_weight, n
    def cb(w, path):
        seen, tail = set(path), []
        for ch in w:
            if ch not in seen:
                tail.append(ch); seen.add(ch)
        before, pairs = _order_metrics(path + tail, w, Lset)
        wt = float(freq[idx[w]]) * scale
        tot[0] += wt * before; tot[1] += wt * pairs; tot[2] += wt; tot[3] += 1
    _each_word_order(s, cb)
    return tot

def guess_order_stats(letters, lengths=range(1, 21), *, strategy=STRATEGY,
                      weighting=WEIGHTING, budget=BUDGET, fill_capped=FILL_CAPPED):
    """Per-length averages of two guess-order quantities for the collection `letters`,
    under the completion convention (see markdown):
      set_before_outside        -- set-letter OCCURRENCES revealed before the first
                                    outside letter is guessed (request 1);
      pairs_outside_before_set  -- pairs (alpha in set, beta not, beta before alpha),
                                    plus the never-guessed-alpha rule (request 2).
    Averaged over the `weighting` distribution -- each word weighted by its frequency,
    cross-length-normalised to N_L per length (common branches count proportionally;
    == uniform-by-word when weighting='uniform'). ONE traversal of the decision tree
    per length; one row per length; also prints the all-lengths OVERALL for both.
    `strategy` is free (default optimal)."""
    Lset = {c for c in letters.lower() if c.isalpha()}
    rows, tot_before, tot_pairs, tot_weight, tot_n = [], 0.0, 0.0, 0.0, 0
    for L in lengths:
        s = sview.make_strategy(strategy, L, weighting=weighting, budget=budget,
                                fill_capped=fill_capped)
        if not s.available:
            continue
        sb, pr, sw, nW = _accumulate(s, Lset)
        rows.append({'length': L, 'n_words': nW,
                     'set_before_outside': round(sb / sw, 4) if sw else 0.0,
                     'pairs_outside_before_set': round(pr / sw, 4) if sw else 0.0})
        tot_before += sb; tot_pairs += pr; tot_weight += sw; tot_n += nW
    if not tot_weight:
        print(f'no committed solve at any length  [{strategy} / {weighting}]')
        return pd.DataFrame()
    shown = ''.join(sorted(Lset))
    print(f"letters = {shown!r}  |  {strategy} / {weighting}  |  {tot_n:,} words")
    print(f"OVERALL (all lengths): set_before_outside = {tot_before / tot_weight:.4f}, "
          f"pairs_outside_before_set = {tot_pairs / tot_weight:.4f}")
    return pd.DataFrame(rows)

ORDER_LETTERS = 'eiasrtnoldcopugmbhfyvwkxzjq'   # <-- the collection of letters (the "set")
display(guess_order_stats(ORDER_LETTERS))

letters = 'abcdefghijklmnopqrstuvwxyz'  |  optimal / uniform  |  34,483 words
OVERALL (all lengths): set_before_outside = 7.7743, pairs_outside_before_set = 0.0000


,length,n_words,set_before_outside,pairs_outside_before_set
0,1,2,1.0,0.0
1,2,49,2.0,0.0
2,3,640,3.0,0.0
3,4,2120,4.0,0.0
4,5,3495,5.0,0.0
5,6,4824,6.0,0.0
6,7,5541,7.0,0.0
7,8,5267,8.0,0.0
8,9,4505,9.0,0.0
9,10,3359,10.0,0.0


## Greedily grow the set to minimise outside-before-set pairs

Starting from the empty set, repeatedly add the outside letter that **minimises
`pairs_outside_before_set`** (all lengths, using the never-guessed rule above — so rare
letters are *not* favoured), until all 26 letters are in. Each step prints the added
letter, the resulting set, that set's `set_before_outside` and `pairs_outside_before_set`,
and **`top2_diff`** — the runner-up candidate's pairs minus the chosen letter's, i.e. how
decisively the winner beat the next-best option (NaN at the final step). Ties are broken
alphabetically.

Like the table above, the pairs are averaged (and minimised) over the **`WEIGHTING`
distribution** — each word by its cross-length effective weight, so common branches count
proportionally (identical to uniform-by-word when `WEIGHTING = 'uniform'`). Uses the
current `STRATEGY`/`WEIGHTING`.

Because the strategy's guess order per word doesn't depend on the set, every word's
order is collected **once** (one tree traversal per length) into padded arrays, and each
candidate set is scored by a fast vectorised pass — the whole 26-step run is ~2 s.

In [11]:
def _collect_word_orders(*, strategy=STRATEGY, weighting=WEIGHTING, budget=BUDGET,
                         fill_capped=FILL_CAPPED, lengths=range(1, 21)):
    """Every word's guess order over `lengths`, as padded arrays (computed ONCE; the
    order does not depend on the set). Returns (order_mat, occ_mat, valid_len, ew, n):
    order_mat[i] = word i's order as letter indices 0..25 (padded with 26),
    occ_mat[i]   = occurrences of each order letter in word i, valid_len[i] = length,
    ew[i]        = word i's cross-length effective weight freq * N_L/total_freq_L
                   (sums to N_L per length; all-ones under weighting='uniform')."""
    recs = []
    for L in lengths:
        s = sview.make_strategy(strategy, L, weighting=weighting, budget=budget,
                                fill_capped=fill_capped)
        if not s.available:
            continue
        freq = s.word_freq()
        total = float(freq.sum())
        scale = (s.g.N / total) if total else 0.0
        idx = s.g._index
        def cb(w, path):
            seen, tail = set(path), []
            for ch in w:
                if ch not in seen:
                    tail.append(ch); seen.add(ch)
            order = path + tail
            recs.append(([ord(c) - 97 for c in order], [w.count(c) for c in order],
                         float(freq[idx[w]]) * scale))
        _each_word_order(s, cb)
    n = len(recs)
    maxlen = max((len(o) for o, _, _ in recs), default=0)
    order_mat = np.full((n, maxlen), 26, dtype=np.int64)
    occ_mat = np.zeros((n, maxlen), dtype=np.int64)
    valid_len = np.zeros(n, dtype=np.int64)
    ew = np.zeros(n, dtype=np.float64)
    for i, (o, occ, w) in enumerate(recs):
        order_mat[i, :len(o)] = o
        occ_mat[i, :len(occ)] = occ
        valid_len[i] = len(o)
        ew[i] = w
    return order_mat, occ_mat, valid_len, ew, n

def greedy_min_pairs(*, strategy=STRATEGY, weighting=WEIGHTING, budget=BUDGET,
                     fill_capped=FILL_CAPPED, lengths=range(1, 21)):
    """Greedily add one letter at a time -- each step the outside letter that MINIMISES
    pairs_outside_before_set (never-guessed rule) -- until all 26 are in. Prints the
    added letter, the set, set_before_outside, pairs_outside_before_set, and top2_diff
    (runner-up's pairs minus the chosen letter's; NaN at the last step). Averaged over
    the `weighting` distribution (each word by its cross-length effective weight;
    == uniform-by-word when weighting='uniform'). Returns the df."""
    order_mat, occ_mat, valid_len, weight, nW = _collect_word_orders(
        strategy=strategy, weighting=weighting, budget=budget,
        fill_capped=fill_capped, lengths=lengths)
    W = float(weight.sum())                            # total weight (== nW under uniform)
    maxlen = order_mat.shape[1]
    pos = np.arange(maxlen)
    valid = pos[None, :] < valid_len[:, None]
    in_L = np.zeros(27, dtype=bool)                    # index 26 = padding sentinel

    def pairs_total(inL):
        inL_seq = inL[order_mat]
        out_seq = valid & ~inL_seq
        before_out = np.cumsum(out_seq, axis=1) - out_seq
        old = (inL_seq * before_out).sum(axis=1)                # outside-before-alpha (guessed alphas)
        n_out = out_seq.sum(axis=1)                             # outside letters guessed
        k_missing = int(inL[:26].sum()) - inL_seq.sum(axis=1)   # in-set letters never guessed
        return float((weight * (old + k_missing * n_out)).sum())

    def before_total(inL):
        inL_seq = inL[order_mat]
        out_seq = valid & ~inL_seq
        has_out = out_seq.any(axis=1)
        first_out = np.where(has_out, out_seq.argmax(axis=1), valid_len)
        before_first = pos[None, :] < first_out[:, None]
        return float((weight * (occ_mat * before_first).sum(axis=1)).sum())

    rows, chosen, remaining = [], [], list(range(26))
    print(f"greedy min pairs_outside_before_set  |  {strategy} / {weighting}  |  {nW:,} words")
    for step in range(26):
        scored = []
        for x in remaining:                            # pairs total for each outside candidate
            in_L[x] = True
            scored.append((pairs_total(in_L), x))
            in_L[x] = False
        scored.sort()                                  # (pairs, letter) asc -> ties alphabetical
        best_tot, best_x = scored[0]
        second_tot = scored[1][0] if len(scored) > 1 else None
        in_L[best_x] = True
        remaining.remove(best_x); chosen.append(best_x)
        p_avg = best_tot / W
        gap = (second_tot - best_tot) / W if second_tot is not None else np.nan
        b_avg = before_total(in_L) / W
        letter = chr(97 + best_x)
        setstr = ''.join(sorted(chr(97 + c) for c in chosen))
        rows.append({'step': step + 1, 'added': letter, 'set': setstr,
                     'set_before_outside': round(b_avg, 4),
                     'pairs_outside_before_set': round(p_avg, 4),
                     'top2_diff': round(gap, 4) if second_tot is not None else np.nan})
        print(f"  step {step + 1:2d}: +{letter}   set={setstr:<26}  "
              f"set_before_outside={b_avg:7.4f}   pairs={p_avg:7.4f}   top2_diff={gap:7.4f}")
        if second_tot is not None and gap > 0.5:
            print()
    return pd.DataFrame(rows)

sel = greedy_min_pairs()
display(sel)

greedy min pairs_outside_before_set  |  optimal / uniform  |  34,483 words
  step  1: +e   set=e                           set_before_outside= 0.7848   pairs= 0.5825   top2_diff= 3.4313

  step  2: +i   set=ei                          set_before_outside= 1.2231   pairs= 3.6097   top2_diff= 0.1647
  step  3: +a   set=aei                         set_before_outside= 1.5078   pairs= 5.8862   top2_diff= 0.9327

  step  4: +s   set=aeis                        set_before_outside= 1.7425   pairs= 8.1713   top2_diff= 0.1123
  step  5: +r   set=aeirs                       set_before_outside= 2.0336   pairs= 9.6866   top2_diff= 0.7284

  step  6: +t   set=aeirst                      set_before_outside= 2.3189   pairs=11.0612   top2_diff= 0.0339
  step  7: +n   set=aeinrst                     set_before_outside= 2.7842   pairs=11.6554   top2_diff= 0.3365
  step  8: +o   set=aeinorst                    set_before_outside= 3.2345   pairs=11.8137   top2_diff= 0.8025

  step  9: +l   set=aeilnorst    

,step,added,set,set_before_outside,pairs_outside_before_set,top2_diff
0,1,e,e,0.7848,0.5825,3.4313
1,2,i,ei,1.2231,3.6097,0.1647
2,3,a,aei,1.5078,5.8862,0.9327
3,4,s,aeis,1.7425,8.1713,0.1123
4,5,r,aeirs,2.0336,9.6866,0.7284
5,6,t,aeirst,2.3189,11.0612,0.0339
6,7,n,aeinrst,2.7842,11.6554,0.3365
7,8,o,aeinorst,3.2345,11.8137,0.8025
8,9,l,aeilnorst,3.4786,12.0438,0.4202
9,10,d,adeilnorst,3.8679,12.0344,0.3315
